In [2]:
%pip install requests

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [requests]
Note: you may need to restart the kernel to use updated packages.


In [3]:
from __future__ import annotations
 
from dataclasses import dataclass, field, asdict
from datetime import date, datetime
from typing import Optional, List, Literal, Union, Any, Dict
 
import requests
 
BASE_URL = "http://172.16.1.210:10000"
SEARCH_ENDPOINT = f"{BASE_URL}/patents/search"
 
 
# 요청 payload를 만들기 위한 dataclass들 (서버의 Pydantic 모델과 필드명을 그대로 맞춰야 함)
 
DateField = Literal[
    "application_date",
    "registration_date",
    "publication_date",
    "int_application_date",
    "int_publication_date",
    "exam_date",
]
KeywordTarget = Literal["office_action", "opinion", "amendment"]
# office_action = office_action = 의견제출통지서 / opinion = 의견서 / amendment = 보정서
KeywordOperator = Literal["AND", "OR", "NOT"]
 
 
@dataclass
class DateRangeFilter:
    field: DateField
    date_from: Optional[date | datetime] = None
    date_to: Optional[date | datetime] = None
 
    def to_payload(self) -> Dict[str, Any]:
        payload = {"field": self.field}
        # 서버 모델이 alias("from"/"to")로 받으므로 그 이름에 맞춰 보냄
        if self.date_from is not None:
            payload["from"] = _iso(self.date_from)
        if self.date_to is not None:
            payload["to"] = _iso(self.date_to)
        return payload
 
 
@dataclass
class KeywordFilter:
    query: str
    targets: List[KeywordTarget] = field(default_factory=lambda: ["office_action", "opinion", "amendment"])
    operator: KeywordOperator = "AND"
 
    def to_payload(self) -> Dict[str, Any]:
        return {"query": self.query, "targets": self.targets, "operator": self.operator}
 
 
@dataclass
class IpcFilter:
    section: Optional[str] = None
    class_code: Optional[str] = None
    subclass: Optional[str] = None
    main_group: Optional[str] = None
    subgroup: Optional[str] = None
 
    def to_payload(self) -> Dict[str, Any]:
        return {k: v for k, v in asdict(self).items() if v is not None}
 
 
@dataclass
class StatuteFilter:
    law_type: Optional[Union[int, str]] = None  # 예: 1 또는 "특허법"
    article: Optional[int] = None
    paragraph: Optional[int] = None
    sub_paragraph: Optional[int] = None
 
    def to_payload(self) -> Dict[str, Any]:
        return {k: v for k, v in asdict(self).items() if v is not None}
 
 
@dataclass
class PatentSearchFilters:
    legal_status_text: Optional[List[str]] = None     # 예: ["등록", "공개"]
    exam_status_text: Optional[List[str]] = None       # 예: ["심사중"]
    exam_requested: Optional[bool] = None
    attorney_names: Optional[List[str]] = None      # 예: ["홍길동"]  (attorney_name)
    examiner_names: Optional[List[str]] = None          # 예: ["김심사"]  (examiner.name)
    has_opinion: Optional[bool] = None
    has_amendment: Optional[bool] = None
    ipc: Optional[List[IpcFilter]] = None
    statutes: Optional[List[StatuteFilter]] = None
    date_ranges: Optional[List[DateRangeFilter]] = None
 
    def to_payload(self) -> Dict[str, Any]:
        payload: Dict[str, Any] = {}
        if self.legal_status_text:
            payload["legal_status_text"] = self.legal_status_text
        if self.exam_status_text:
            payload["exam_status_text"] = self.exam_status_text
        if self.exam_requested is not None:
            payload["exam_requested"] = self.exam_requested
        if self.attorney_names:
            payload["attorney_names"] = self.attorney_names
        if self.examiner_names:
            payload["examiner_names"] = self.examiner_names
        if self.has_opinion is not None:
            payload["has_opinion"] = self.has_opinion
        if self.has_amendment is not None:
            payload["has_amendment"] = self.has_amendment
        if self.ipc:
            payload["ipc"] = [i.to_payload() for i in self.ipc]
        if self.statutes:
            payload["statutes"] = [s.to_payload() for s in self.statutes]
        if self.date_ranges:
            payload["date_ranges"] = [d.to_payload() for d in self.date_ranges]
        return payload
 
 
def _iso(value: date | datetime) -> str:
    if isinstance(value, datetime):
        return value.isoformat()
    return value.isoformat()  # date도 isoformat() 있음 (YYYY-MM-DD)
 
 
# =========================================================
# 실제 API 호출 함수
# =========================================================
 
def search_patents(
    filters: Optional[PatentSearchFilters] = None,
    keywords: Optional[List[KeywordFilter]] = None,
    page: int = 1,
    size: int = 20,
    timeout: float = 30.0,
) -> Dict[str, Any]:
    """
    /patents/search를 호출하고 결과 dict({"total", "page", "size", "data"})를 반환합니다.
    필터를 아예 넘기지 않으면(filters=None, keywords=None) 조건 없이 전체 조회합니다.
    """
    body: Dict[str, Any] = {
        "page": page,
        "size": size,
    }
 
    if filters is not None:
        filter_payload = filters.to_payload()
        if filter_payload:
            body["filters"] = filter_payload
 
    if keywords:
        body["keywords"] = [k.to_payload() for k in keywords]
 
    resp = requests.post(SEARCH_ENDPOINT, json=body, timeout=timeout)
 
    if resp.status_code != 200:
        # 서버가 500이면 detail에 에러 메시지가 들어있으므로 그대로 노출
        raise RuntimeError(f"검색 실패 ({resp.status_code}): {resp.text}")
 
    return resp.json()


/Users/vrn-jisooyun/Desktop/VRN_DEV/myWorkspace/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### 사용 예시

In [5]:
filters = PatentSearchFilters(has_opinion=True, has_amendment=True)
keywords = [
    KeywordFilter(query="egfr", targets=["office_action"]),
    KeywordFilter(query="egfr", targets=["opinion"], operator="AND"),
]

body = {
    "page": 1,
    "size": 20,
    "filters": filters.to_payload(),
    "keywords": [k.to_payload() for k in keywords],
}

print("SEARCH_ENDPOINT:", SEARCH_ENDPOINT)
print("REQUEST BODY:", body)

try:
    resp = requests.post(SEARCH_ENDPOINT, json=body, timeout=30)
    print("STATUS:", resp.status_code)
    print("RESPONSE TEXT:", resp.text[:1000])  # 너무 길면 앞부분만
    resp.raise_for_status()
    result = resp.json()
    result
except Exception as e:
    print(type(e).__name__, e)

SEARCH_ENDPOINT: http://172.16.1.210:10000/patents/search
REQUEST BODY: {'page': 1, 'size': 20, 'filters': {'has_opinion': True, 'has_amendment': True}, 'keywords': [{'query': 'egfr', 'targets': ['office_action'], 'operator': 'AND'}, {'query': 'egfr', 'targets': ['opinion'], 'operator': 'AND'}]}
STATUS: 200
RESPONSE TEXT: {"total":96,"page":1,"size":20,"data":[{"office_action_id":11850,"admin_id":108229,"office_action_content":"발송번호: 9-5-2023-097003752 발송일자: 2023.10.26. 제출기일: 2023.12.26.\n\n수신 :\n\n특 허 청\n\n# 의견제출통지서\n\n출 원 인 성 명 주식회사 파이안바이오테크놀로지 (특허고객번호:\n\n120140198303) 주 소 대 리 인 성 명 류종우 외 2 명 주 소\n\n발 명 자 성 명 김천형 주 소 발 명 자 성 명 한규범 주 소 발 명 자 성 명 유신혜 주 소 발 명 자 성 명 김유진 주 소 발 명 자 성 명 이서은 주 소 발 명 자 성 명 박종혁 주 소 발 명 자 성 명 조가영 주 소 발 명 자 성 명 강영철 주 소\n\n출 원 번 호 10-2021-0141668 출 원 일 자 2021.10.22. 발 명 의 명 칭 항암제를 포함한 미토콘드리아 및 이의 용도\n\n- 1. 이 출원에 대한 심사결과 다음과 같은 거절이유가 있어 특허법 제63조에 따라 이를 통지 하오니 의견이 있거나 보정이 필요할 경우에는 상기 제출기일(2023.12.26.)까지 의견(답변, 소명)서[특허법시행규칙 별지 제24호서식] 또는/및 보정서[특허법시행규칙 별지 제9호서식] 를 